In [1]:
import pandas as pd
import re
df = pd.read_csv('structed_data.csv', encoding='utf-8')

In [2]:
# 岗位类型，One-hot聚类

# ===== 1. 关键词词库=====
ROLE_DICT = {
    "DA": [
        "数据アナリスト", "データ分析", "データ集計", "マーケティングデータ",
        "ビジネスアナリスト", "Business Analyst", "データ解析", "インサイト分析",
        "顧客データ", "アナリティクス", "データ運用" 
    ],
    "DS": [
        "データサイエンティスト", "数理", "AI", "機械学習", 
        "Machine Learning", "LLM", "ゲノム解析", "画像解析", 
        "動作解析", "深層学習", "統計" 
    ],
    "DE": [
        "データエンジニア", "基盤", "構築", "DWH", "ETL", "Databricks",
        "データプラットフォーム", "Data Platform", "データ基盤", 
        "パイプライン", "アナリティクスエンジニア" 
    ],
    "MLE": [
        "機械学習エンジニア", "MLエンジニア", "AIエンジニア", 
        "ML Engineer", "Software Engineer(ML)" 
    ],
    "BI": [
        "BI", "ダッシュボード", "可視化", "Tableau", "Looker", 
        "PowerBI", "レポート作成" 
    ],
    "CONSULTANT": [
        "コンサル", "利活用", "データ活用", "戦略", "DX推進", 
        "マーケティング戦略", "企画", "調査", "リサーチ", "Research",
        "マーケター", "マーケティング担当", "市場開発", "Promotion",
        "プロダクトマネージャー", "事業開発", "PMM", "BizOps", "RevOps" 
    ]
}

# ===== 2. 优先级（保持不变，确保交叉学科时的准确性）=====
PRIORITY = ["DS", "MLE", "DE", "DA", "BI", "CONSULTANT"]

# ===== 3. 分类函数 (优化了大小写匹配) =====
def classify_role(title):
    if not isinstance(title, str):
        return "OTHER"
    
    # 统一转大写，处理如 "bi" 和 "BI" 的匹配问题
    title_upper = title.upper()
    matched_roles = []

    for role, keywords in ROLE_DICT.items():
        for kw in keywords:
            if kw.upper() in title_upper:
                matched_roles.append(role)
                break

    if not matched_roles:
        return "OTHER"

    # 按优先级返回最高的一个
    for p in PRIORITY:
        if p in matched_roles:
            return p

# ===== 4. 应用与验证 =====
df["role_type"] = df["Position"].apply(classify_role)

print("\n=== 岗位分布 ===")
print(df["role_type"].value_counts())


=== 岗位分布 ===
role_type
OTHER         381
CONSULTANT    380
DS            290
DE            101
DA             90
BI             11
Name: count, dtype: int64


In [3]:
# 在宅情况，One-hot聚类

# ===== 1. 关键词配置（保持你的逻辑，增强覆盖面）=====
REMOTE_KW = ["リモート", "在宅", "テレワーク", "フルリモート"]
ALLOW_KW = ["可", "可能", "OK", "利用", "対応", "推奨", "導入", "活用", "併用"]
NEGATIVE_REMOTE = ["不可", "実施しておりません", "対象外", "できません"]
ONSITE_KW = ["出社", "常駐", "対面"]

# ===== 2. 预处理函数 =====
def normalize_text(text):
    if not isinstance(text, str): return ""
    # 显式处理代码1生成的“无匹配”
    if text == "无匹配": return "NONE"
    # 去掉空格和换行，但保留 || 作为逻辑分隔（或者直接去掉也可）
    text = re.sub(r"\s+", "", text)
    return text

# ===== 3. 分类函数（保持判断逻辑，优化匹配深度）=====
def classify_remote(text):
    text = normalize_text(text)
    
    if text == "NONE" or text == "":
        return "UNKNOWN", -1

    # ---- ① 否定优先（针对远程词的否定） ----
    # 捕捉类似 “在宅不可”、“リモートワークは実施しておりません”
    for r_kw in REMOTE_KW:
        if r_kw in text:
            for n_kw in NEGATIVE_REMOTE:
                if n_kw in text and text.find(r_kw) < text.find(n_kw) < text.find(r_kw) + 10:
                    return "ONSITE", 0

    # ---- ② 强判定：组合命中（你的核心逻辑） ----
    for r_kw in REMOTE_KW:
        if r_kw in text:
            for a_kw in ALLOW_KW:
                if a_kw in text:
                    return "REMOTE", 1

    # ---- ③ 弱判定：补充命中（针对“一部従業員利用可”或“フルリモート”等固定搭配） ----
    # 很多文本里“可”在括号里，上面的逻辑有时会漏掉，这里做一层兜底
    if any(kw in text for kw in ["フルリモート", "リモートワーク制度", "在宅勤務制度"]):
        return "REMOTE", 1

    # ---- ④ 明确 onsite ----
    # 如果前面没匹配到远程许可，且出现了出社关键词
    for kw in ONSITE_KW:
        if kw in text:
            return "ONSITE", 0

    # ---- ⑤ 最后的兜底：含有远程词但无明确许可/否定 ----
    # 只要出现了在宅相关的词，在招聘语境下大概率是支持的，减少 UNKNOWN
    if any(kw in text for kw in REMOTE_KW):
        return "REMOTE", 1

    return "UNKNOWN", -1

# ===== 4. 应用 =====
# 假设 df 是你的原始 DataFrame
res = df["在宅可否"].apply(lambda x: pd.Series(classify_remote(x)))
df["remote_flag"] = res[0]
df["remote_score"] = res[1]

# ===== 5. 分布 =====
print("\n=== remote_flag分布 ===")
print(df["remote_flag"].value_counts())


=== remote_flag分布 ===
remote_flag
REMOTE     940
UNKNOWN    270
ONSITE      43
Name: count, dtype: int64


In [4]:
# 年收进一步分类
def extract_salary(salary_text):
    if pd.isna(salary_text) or salary_text == "":
        return "UNKNOWN", "UNKNOWN", "UNKNOWN"
    
    # 统一转为字符串并去掉空格
    text = str(salary_text).replace(" ", "").replace(",", "")
    
    # 正则表达式：匹配 500万円～900万円 这种格式
    # 允许匹配：500万, 500万円, 500~900, 500-900 等变体
    pattern = r'(\d{3,4})(?:万円|万)?(?:～|-|~)(\d{3,4})(?:万円|万)?'
    match = re.search(pattern, text)
    
    if match:
        try:
            s_min = int(match.group(1))
            s_max = int(match.group(2))
            s_avg = (s_min + s_max) / 2
            return s_min, s_max, s_avg
        except ValueError:
            return "UNKNOWN", "UNKNOWN", "UNKNOWN"
    
    # 兜底逻辑：如果只有单值（如 500万円～）
    single_pattern = r'(\d{3,4})万円'
    single_match = re.search(single_pattern, text)
    if single_match:
        try:
            s_min = int(single_match.group(1))
            return s_min, "UNKNOWN", "UNKNOWN"
        except ValueError:
            pass

    return "UNKNOWN", "UNKNOWN", "UNKNOWN"

# ===== 应用清洗 =====
# 假设你的 DataFrame 叫 df，“给与”列是年收所在列
salary_data = df["給与"].apply(extract_salary)

# 将结果展开到三列
df[['salary_min', 'salary_max', 'salary_avg']] = pd.DataFrame(salary_data.tolist(), index=df.index)

# ===== 打印结果查看 =====
print(df[['給与', 'salary_min', 'salary_max', 'salary_avg']].head())

                                                  給与 salary_min salary_max  \
0  想定年収 630万円～810万円 賃金 月給制 月給 525,000円～ 月給￥525,00...        630        810   
1  想定年収 610万円～910万円 賃金 月給制 月給 267,000円～ 月給￥267,00...        610        910   
2  想定年収 500万円～700万円 賃金 月給制 月給 312,500円～ 月給￥312,50...        500        700   
3  想定年収 590万円～690万円 賃金 月給制 月給 330,000円～390,000円 月...        590        690   
4  想定年収 440万円～570万円 賃金 月給制 月給 250,000円～320,000円 月...        440        570   

  salary_avg  
0      720.0  
1      760.0  
2      600.0  
3      640.0  
4      505.0  


In [ ]:

def extract_experience_years_v2(text):
    if pd.isna(text) or text == "" or text == "无匹配":
        return 0
    
    text = str(text)
    
    # --- 1. 优先提取具体年数 ---
    # 匹配：3年以上、5年以上の実務経験、经验2年 等
    year_match = re.search(r'(\d{1,2})\s*年以上?|経験(\d{1,2})\s*年', text)
    if year_match:
        # 取出匹配到的第一个非空数字组
        val = year_match.group(1) if year_match.group(1) else year_match.group(2)
        return int(val)
    
    # 匹配：社会人年次2年目～5年目 (取最小值)
    genji_match = re.search(r'社会人年次(\d{1,2})年目', text)
    if genji_match:
        return int(genji_match.group(1))

    # --- 2. 逻辑调整：必须有项目经验但未写年数的设为 1 ---
    # 定义表示“有经验”的关键词
    has_exp_keywords = [
        "経験", "実務", "開発経験", "運用経験", "構築経験", 
        "携わった", "職務での", "スキルをお持ちの方"
    ]
    
    if any(kw in text for kw in has_exp_keywords):
        return 1

    # --- 3. 彻底没提到的设为 0 ---
    return 0

# ===== 应用清洗 =====
df["experience_years"] = df["必要経験"].apply(extract_experience_years_v2)


# 查看分布
print("\n=== 经验年数分布 ===")
print(df["experience_years"].value_counts())


=== 经验年数分布 ===


In [6]:


def classify_level(row):
    # 获取 Position 和之前清洗出的经验年数
    position = str(row['Position']).upper()
    exp = row['experience_years']
    
    # ===== 1. MANAGER (管理/领导层) =====
    # 关键词包含：经理、组长、PM、负责人、管理监督者等
    manager_keywords = [
        "マネージャー", "MANAGER", "リーダー", "LEADER", "PM", "PMO", "PL", "CMO",
        "室長", "課長", "部長", "責任者", "幹部", "管理监督者", "管理職", "パートナー"
    ]
    if any(kw in position for kw in manager_keywords):
        return "manager"

    # ===== 2. SENIOR (资深/专家/上流工程) =====
    # 关键词包含：资深、专家、首席、架构师、设计、战略、以及 3 年以上经验
    senior_keywords = [
        "リード", "LEAD", "シニア", "SENIOR", "エキスパート", "EXPERT", 
        "設計", "上流", "アーキテクト", "ARCHITECT", "戦略", "エバンジェリスト",
        "プロ採用"
    ]
    if any(kw in position for kw in senior_keywords) or exp >= 3:
        return "senior"

    # ===== 3. JUNIOR (初级/潜力/未开发) =====
    # 关键词包含：未经验、潜力、研修、第二新卒、或是经验为 0 且不含管理词
    junior_keywords = [
        "未経験", "ポテンシャル", "研修", "第二新卒", "アシスタント", 
        "新卒", "オープンポジション", "OPEN POSITION", "メンバークラス"
    ]
    if any(kw in position for kw in junior_keywords) or exp == 0:
        return "junior"

    # ===== 4. MID (中坚/中级) =====
    # 默认分类：有一定实务经验（1-3年），且职位没有明确标注为管理或资深
    return "mid"

# ===== 应用函数 =====
df['level'] = df.apply(classify_level, axis=1)

# ===== 打印分布情况 =====
print("\n=== Level 分布统计 ===")




=== Level 分布统计 ===


In [2]:
drop_cols = ['序号', '更新日', 'Position', '会社名', '従業員数', '在宅可否', 
             '給与',  '配属部门', '就業時間', '選考内容', '清洗后全文']

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [ ]:
# 筛选关键词
from janome.tokenizer import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 初始化 Janome 分词器
t = Tokenizer()

# 2. 定义分词函数：只保留名词 (Noun)
def janome_tokenizer(text):
    tokens = []
    # 分词并过滤掉无意义的词性
    for token in t.tokenize(text):
        # part_of_speech 格式如: "名詞,一般,*,*"
        pos = token.part_of_speech.split(',')[0]
        if pos == '名詞':
            # 过滤掉单字名词（通常是无意义的符号或数字）
            if len(token.surface) > 1:
                tokens.append(token.surface)
    return tokens

# 3. 准备数据 (假设 df 是你的数据源)
# data = df['必要経験'].fillna('').tolist()
# 这里用你提供的那一段数据做演示：
texts = df['必要経験'].fillna('').tolist()

# 4. 配置 TF-IDF
# stop_words 可以加入你认为无意义的词，比如 '経験', '務' 等
vectorizer = TfidfVectorizer(
    tokenizer=janome_tokenizer,
    token_pattern=None,
    max_df=0.8,    # 过滤掉在 80% 以上岗位都出现的通用词
    min_df=2,      # 词项至少出现 2 次
    max_features=100 # 只取前 100 个特征词
)

# 5. 运行计算
tfidf_matrix = vectorizer.fit_transform(texts)

# 6. 整理结果
feature_names = vectorizer.get_feature_names_out()
weights = tfidf_matrix.toarray().mean(axis=0)

keyword_df = pd.DataFrame({'keyword': feature_names, 'tfidf_weight': weights})
top_keywords = keyword_df.sort_values(by='tfidf_weight', ascending=False)

keyword_df.to_csv('keyword.csv', index=False, encoding='utf-8-sig')


--- Janome 分词版：核心关键词权重排名 ---
    keyword  tfidf_weight
20      データ      0.114876
40       分析      0.088173
93       開発      0.082223
64       業務      0.078629
31       事業      0.073397
32       以上      0.072454
26  マーケティング      0.070360
47       実務      0.064751
97       高専      0.061494
68       歓迎      0.060709
8        こと      0.059563
74       短大      0.058128
50       専修      0.057769
34       企画      0.056971
98       高校      0.056252
0        ai      0.052553
65       業界      0.051530
99       魅力      0.051306
86       設計      0.050874
71       環境      0.050856
91       運用      0.049528
16      スキル      0.049136
22   プロジェクト      0.048052
69       活用      0.046765
44       営業      0.046006
73       知識      0.045486
43       可能      0.044698
58       技術      0.044480
82       英語      0.043617
35       会社      0.042792
66       構築      0.042262
56       成長      0.042179
7       いずれ      0.042084
18      ツール      0.040248
57       戦略      0.039950
15     システム      0.038746
14     サー

In [ ]:
# 建立关键词字典
keywords = [
    "Python", "SQL", "R", "C++", "Java", "MATLAB", "PyTorch", "TensorFlow", "scikit-learn",
    "機械学習", "AI", "データ分析", "データサイエンス", "統計解析", "深層学習", "ディープラーニング",
    "LLM", "生成AI", "自然言語処理", "画像処理", "信号処理",
    "AWS", "Azure", "GCP", "クラウド", "DWH", "ETL", "BIツール", "Tableau", "Power BI",
    "英語", "TOEIC", "ビジネスコミュニケーション", "マーケティング", "要件定義", "設計"
]

# 执行 One-Hot 编码
for word in keywords:
    # 创建新列：如果【必要経験】中包含该词，则记为 1，否则为 0
    # na=False 处理空值，case=False 忽略英文字母大小写
    df_processed[f'exp_{word}'] = df['必要経験'].str.contains(word, case=False, na=False).astype(int)



NameError: name 'df_processed' is not defined

In [ ]:

# 1. 筛选出所有以 'exp_' 开头的特征列
exp_cols = [f'exp_{word}' for word in keywords]

# 2. 计算各项指标
# .sum() 统计 1 的个数（出现次数）
# .mean() 统计 1 的占比（出现频率，即百分比）
summary_df = pd.DataFrame({
    '出现次数': df_processed[exp_cols].sum(),
    '出现频率 (%)': (df_processed[exp_cols].mean() * 100).round(2)
})

# 3. 按出现次数从高到低排序，让你一眼看到“最热门技能”
summary_df = summary_df.sort_values(by='出现次数', ascending=False)

print("--- 技能关键词统计分析 ---")
print(summary_df)

# 4. 可选：保存到 Excel 或 CSV 方便分析
# summary_df.to_csv("skill_analysis.csv", encoding='utf_8_sig')

In [ ]:
drop_cols = ['必要経験', '仕事内容']

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [9]:
df.to_csv('one_hot.csv', index=False, encoding='utf-8-sig')